# Entity History v1 — Training with Event History Encoder

**Notebook Version**: 1.0  
**Purpose**: Train `entity_history_v1` — the first entity model with an explicit event-history attention encoder.  
**Research question**: *Can selective retention of historical battle events compensate for the information loss of fixed-size state representations?*

Phases:
1. **Setup** — GPU verification, cost baseline, deps, data
2. **Train** — entity model with history encoder + sequence + value auxiliary heads
3. **Cost Report** — GPU-hours, Colab CU estimate, per-epoch time
4. **Attention Analysis** — attention weight distributions over history turns (semi-observed execution)
5. **Diagnostics** — loss curves, history buffer utilization, event-category breakdown
6. **Push** — GCS artifact upload

**Ablation position**: Cell B and C of the 3-cell ablation (vs `entity_action_v2_20260409_1811` as Cell A).

## Phase 0: GPU Verification & Cost Baseline

In [ ]:
# ── 0. GPU verify + cost baseline ──────────────────────────────────────────
import subprocess, time, json, os

TRAINING_START_WALL = time.time()

# GPU identity
gpu_info = {}
try:
    r = subprocess.run(
        ['nvidia-smi',
         '--query-gpu=name,memory.total,power.limit,driver_version',
         '--format=csv,noheader'],
        capture_output=True, text=True, timeout=5
    )
    if r.returncode == 0:
        parts = [p.strip() for p in r.stdout.strip().split(',')]
        gpu_info = {
            'name':           parts[0] if len(parts) > 0 else 'unknown',
            'memory_total':   parts[1] if len(parts) > 1 else 'unknown',
            'power_limit_w':  parts[2] if len(parts) > 2 else 'unknown',
            'driver_version': parts[3] if len(parts) > 3 else 'unknown',
        }
except Exception as e:
    gpu_info = {'name': 'CPU', 'error': str(e)}

# Cost estimate map (Colab compute units per GPU-hour, approximate)
GPU_CU_PER_HOUR = {
    'T4':   1.96,
    'A100': 12.25,
    'V100': 6.86,
    'L4':   4.90,
    'CPU':  0.0,
}
gpu_name = gpu_info.get('name', 'CPU')
gpu_tier = next((k for k in GPU_CU_PER_HOUR if k in gpu_name.upper()), 'CPU')
CU_RATE  = GPU_CU_PER_HOUR[gpu_tier]

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')

print('=' * 70)
print('GPU BASELINE')
print('=' * 70)
for k, v in gpu_info.items():
    print(f'  {k:20s}: {v}')
print(f'  TF version          : {tf.__version__}')
print(f'  TF GPUs detected    : {[g.name for g in gpus]}')
print(f'  Cost tier           : {gpu_tier}  ({CU_RATE} CU/hr)')
print(f'  Wall-clock baseline : {time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(TRAINING_START_WALL))}')
print('=' * 70)

if not gpus and gpu_tier == 'CPU':
    print()
    print('WARNING: No GPU — training will be very slow.')
    print('Colab: Runtime → Change runtime type → T4 GPU, then restart.')

In [ ]:
# ── 0b. Keep-alive + GCS checkpoint config ─────────────────────────────────
from IPython.display import Javascript, display

# Prevents Colab from marking the tab idle and disconnecting
display(Javascript("""
  function keepColab() { document.querySelector('#top-toolbar')?.click(); }
  window._keepAlive = setInterval(keepColab, 55000);
  console.log('Keep-alive active. Stop: clearInterval(window._keepAlive)');
"""))
print('Keep-alive active (pings Colab every 55s).')

# GCS prefix for mid-training checkpoints
GCS_CHECKPOINT_PREFIX = 'gs://artifacts-model-serving/checkpoints/entity_history_v1'
print(f'Checkpoint prefix: {GCS_CHECKPOINT_PREFIX}')

## Setup: Dependencies and Data

In [ ]:
# ── 1. Clone repo ──────────────────────────────────────────────────────────
import os, shutil

REPO_URL = 'https://github.com/AlterProgramming/Pokemon-Showdown-Agents-Go-Brrrr.git'
BRANCH   = 'main'
REPO_DIR = '/content/Pokemon-Showdown-Agents-Go-Brrrr'

# Step out before deleting — previous run may have os.chdir'd into REPO_DIR.
os.chdir('/content')

if os.path.exists(REPO_DIR):
    print(f'Removing stale checkout: {REPO_DIR}')
    shutil.rmtree(REPO_DIR)

!git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git log --oneline -3

checks = {
    'HistoryAttention class defined':     ('core/EntityModelV1.py', 'class HistoryAttention'),
    'Attention inlined (no list call)':   ('core/EntityModelV1.py', 'history_attn_Wq'),
    'keras.ops inline pooling':           ('core/EntityModelV1.py', 'ops = _keras().ops'),
    'supports_masking on all layers':     ('core/EntityModelV1.py', 'supports_masking = True'),
    'mask=None on all call()':            ('core/EntityModelV1.py', 'def call(self, inputs, mask=None)'),
    'OOM gc.collect':                     ('train_entity_action.py', 'gc.collect'),
}
for label, (path, token) in checks.items():
    found = token in open(path).read()
    status = '✓' if found else '✗ MISSING'
    print(f'  {status}  {label}')
    if not found:
        raise RuntimeError(f'Fix not found in {path}: {token!r} — clone may have failed')

In [ ]:
# ── 2. Install dependencies ────────────────────────────────────────────────
import importlib, subprocess, tensorflow as tf
print('TF:', tf.__version__)

!pip install -q kagglehub

# Pin keras to a version known to handle compute_mask correctly.
# Older Keras 3.x creates an internal Lambda for mask propagation when
# compute_mask is missing, which fails shape inference on list-input layers.
import keras as _keras_check
_ver = tuple(int(x) for x in _keras_check.__version__.split('.')[:2])
if _ver < (3, 14):
    print(f'Upgrading keras {_keras_check.__version__} → >=3.14.0 ...')
    subprocess.run(['pip', 'install', '-q', 'keras>=3.14.0'], check=True)
    importlib.invalidate_caches()

import keras
print(f'keras {keras.__version__} ✓')
print('Done.')

In [ ]:
# ── 3. Download battle data ────────────────────────────────────────────────
import kagglehub, os, glob

DATASET = 'thephilliplin/pokemon-showdown-battles-gen9-randbats'
print('Downloading', DATASET, '...')
data_path = kagglehub.dataset_download(DATASET)

json_files = glob.glob(os.path.join(data_path, '*.json'))
print(f'Dataset path : {data_path}')
print(f'JSON files   : {len(json_files):,}')

## Training Configuration

In [ ]:
# ── 4. Training configuration ──────────────────────────────────────────────
import os
from datetime import datetime

_ts = datetime.now().strftime('%Y%m%d_%H%M')
MODEL_NAME = f'entity_history_v1_{_ts}'
OUTPUT_DIR = '/content/training_artifacts'
RUN_ID     = MODEL_NAME

# ── Data ──────────────────────────────────────────────────────────────────
MAX_BATTLES   = 50     # GPU SMOKE TEST — change back to 5000 for the real run
VAL_RATIO     = 0.2
SEED          = 42

# ── Optimiser ─────────────────────────────────────────────────────────────
EPOCHS        = 30
BATCH_SIZE    = 256
LEARNING_RATE = 0.001
PATIENCE      = 5

# ── Architecture ──────────────────────────────────────────────────────────
HIDDEN_DIM    = 256
DEPTH         = 3
DROPOUT       = 0.1

# ── Auxiliary heads ───────────────────────────────────────────────────────
PREDICT_VALUE         = True
PREDICT_TURN_SEQUENCE = True
PREDICT_TURN_OUTCOME  = False   # omit transition head to isolate sequence + history

VALUE_WEIGHT         = 0.25
SEQUENCE_WEIGHT      = 0.10
SEQUENCE_HIDDEN_DIM  = 128
MAX_SEQ_LEN          = 32

# ── Event history encoder (NEW) ───────────────────────────────────────────
PREDICT_FROM_HISTORY     = True
HISTORY_TURNS            = 8    # K: past turns to retain
HISTORY_EVENTS_PER_TURN  = 24   # E: max event tokens per turn
HISTORY_EMBED_DIM        = 32
HISTORY_LSTM_DIM         = 64

# ── Return weighting ──────────────────────────────────────────────────────
POLICY_RETURN_WEIGHTING    = 'exp'
POLICY_RETURN_WEIGHT_SCALE = 0.75
POLICY_RETURN_WEIGHT_MIN   = 0.25
POLICY_RETURN_WEIGHT_MAX   = 4.0

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('=' * 70)
print('TRAINING CONFIGURATION — entity_history_v1')
print('=' * 70)
print(f'  Model name    : {MODEL_NAME}')
print(f'  Output dir    : {OUTPUT_DIR}')
print(f'  Max battles   : {MAX_BATTLES}')
print(f'  Epochs        : {EPOCHS}  (patience={PATIENCE})')
print(f'  Batch / LR    : {BATCH_SIZE} / {LEARNING_RATE}')
print(f'  Architecture  : hidden={HIDDEN_DIM}  depth={DEPTH}  dropout={DROPOUT}')
print()
print('  Auxiliary heads:')
print(f'    value        : {PREDICT_VALUE}  (w={VALUE_WEIGHT})')
print(f'    sequence     : {PREDICT_TURN_SEQUENCE}  (w={SEQUENCE_WEIGHT}  lstm={SEQUENCE_HIDDEN_DIM})')
print(f'    history enc  : {PREDICT_FROM_HISTORY}  (K={HISTORY_TURNS}  E={HISTORY_EVENTS_PER_TURN}')
print(f'                   embed={HISTORY_EMBED_DIM}  lstm={HISTORY_LSTM_DIM})')
print('=' * 70)

In [ ]:
# ── 4b. Resume from GCS checkpoint (runs automatically) ────────────────────
import subprocess, os, json as _json

RESUME_FROM_CHECKPOINT = False
resume_checkpoint_path = None

result = subprocess.run(
    ['gsutil', 'ls', GCS_CHECKPOINT_PREFIX + '/'],
    capture_output=True, text=True
)
if result.returncode == 0 and result.stdout.strip():
    prior_runs = sorted([l.strip().rstrip('/') for l in result.stdout.strip().split('\n') if l.strip()])
    latest = prior_runs[-1]
    print(f'Prior checkpoint found: {latest}')
    print('Downloading...')
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    subprocess.run(['gsutil', '-m', 'rsync', '-r', latest + '/', OUTPUT_DIR + '/'],
                   capture_output=True)
    ckpt_files = os.listdir(OUTPUT_DIR) if os.path.exists(OUTPUT_DIR) else []
    training_models = sorted([f for f in ckpt_files if f.startswith('training_model') and f.endswith('.keras')])
    if training_models:
        resume_checkpoint_path = os.path.join(OUTPUT_DIR, training_models[-1])
        RESUME_FROM_CHECKPOINT = True
        print(f'Will resume from: {resume_checkpoint_path}')
    else:
        print('Checkpoint folder found but no training model — starting fresh.')
else:
    print('No prior checkpoint in GCS — starting fresh.')

## Phase 1: Training

In [ ]:
# ── 5. Train ───────────────────────────────────────────────────────────────
import subprocess, sys, os, re, time

REPO_DIR = '/content/Pokemon-Showdown-Agents-Go-Brrrr'
env = os.environ.copy()
env['PYTHONPATH'] = REPO_DIR + os.pathsep + env.get('PYTHONPATH', '')

cmd = [
    sys.executable, '-u', 'train_entity_action.py', data_path,
    '--model-name',   MODEL_NAME,
    '--output-dir',   OUTPUT_DIR,
    '--max-battles',  str(MAX_BATTLES),
    '--val-ratio',    str(VAL_RATIO),
    '--epochs',       str(EPOCHS),
    '--batch-size',   str(BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--patience',     str(PATIENCE),
    '--hidden-dim',   str(HIDDEN_DIM),
    '--depth',        str(DEPTH),
    '--dropout',      str(DROPOUT),
    '--seed',         str(SEED),
]

if PREDICT_VALUE:
    cmd += ['--predict-value', '--value-weight', str(VALUE_WEIGHT)]
if PREDICT_TURN_OUTCOME:
    cmd += ['--predict-turn-outcome']
if PREDICT_TURN_SEQUENCE:
    cmd += [
        '--predict-turn-sequence',
        '--sequence-weight',     str(SEQUENCE_WEIGHT),
        '--sequence-hidden-dim', str(SEQUENCE_HIDDEN_DIM),
        '--max-seq-len',         str(MAX_SEQ_LEN),
    ]
if PREDICT_FROM_HISTORY:
    cmd += [
        '--predict-from-history',
        '--history-turns',           str(HISTORY_TURNS),
        '--history-events-per-turn', str(HISTORY_EVENTS_PER_TURN),
        '--history-embed-dim',       str(HISTORY_EMBED_DIM),
        '--history-lstm-dim',        str(HISTORY_LSTM_DIM),
    ]

cmd += [
    '--policy-return-weighting',    POLICY_RETURN_WEIGHTING,
    '--policy-return-weight-scale', str(POLICY_RETURN_WEIGHT_SCALE),
    '--policy-return-weight-min',   str(POLICY_RETURN_WEIGHT_MIN),
    '--policy-return-weight-max',   str(POLICY_RETURN_WEIGHT_MAX),
]

if RESUME_FROM_CHECKPOINT and resume_checkpoint_path:
    cmd += ['--init-checkpoint', resume_checkpoint_path]
    print(f'  → Resuming from checkpoint: {resume_checkpoint_path}')

EPOCH_HDR = re.compile(r'Epoch\s+\d+/\d+')
STEP_RE   = re.compile(r'^\s*(\d+)/(\d+)')
METRIC_RE = re.compile(r'loss|accuracy|mae|brier', re.IGNORECASE)
ERROR_RE  = re.compile(r'error|traceback|exception|warning|failed|killed', re.IGNORECASE)

epoch_times   = []
_epoch_start  = None

train_wall_start = time.time()
print('Starting training...')
print('Command:', ' '.join(cmd[:4]), '... [+', len(cmd)-4, 'args]')
print('=' * 70)

import threading as _threading
_sync_stop = _threading.Event()

def _gcs_sync():
    import time
    while not _sync_stop.wait(300):
        try:
            subprocess.run(
                ['gsutil', '-m', '-q', 'rsync', '-r',
                 OUTPUT_DIR + '/', GCS_CHECKPOINT_PREFIX + '/' + RUN_ID + '/'],
                capture_output=True, timeout=120
            )
            print(f'  [ckpt] synced → {GCS_CHECKPOINT_PREFIX}/{RUN_ID}/')
        except Exception as _e:
            print(f'  [ckpt] sync error: {_e}')

_sync_thread = _threading.Thread(target=_gcs_sync, daemon=True)
_sync_thread.start()
print('Background GCS checkpoint sync started (every 5 min).')

proc = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, cwd=REPO_DIR, env=env,
)

epoch_total_steps = None
mid_epoch_printed = False
suppressed_lines  = []

for raw_line in proc.stdout:
    line = raw_line.rstrip()
    if not line:
        continue
    if ERROR_RE.search(line):
        print(line); suppressed_lines = []; continue
    if EPOCH_HDR.match(line):
        if _epoch_start is not None:
            epoch_times.append(time.time() - _epoch_start)
        _epoch_start = time.time()
        print(); print(line)
        epoch_total_steps = None; mid_epoch_printed = False
        continue
    step_m = STEP_RE.match(line)
    if step_m and METRIC_RE.search(line):
        current_step = int(step_m.group(1))
        total_steps  = int(step_m.group(2))
        suppressed_lines.append(line)
        if epoch_total_steps != total_steps:
            epoch_total_steps = total_steps
        is_final = (current_step == total_steps)
        is_mid   = not mid_epoch_printed and current_step >= total_steps // 2 and not is_final
        if is_mid:
            pct = int(100 * current_step / total_steps)
            print(f'  [{pct:3d}%] {line.strip()}')
            mid_epoch_printed = True
        elif is_final:
            # Full epoch-end line — includes all val_* metrics
            print(f'  {line.strip()}')
        continue
    suppressed_lines.append(line)

if _epoch_start is not None:
    epoch_times.append(time.time() - _epoch_start)

proc.wait()
_sync_stop.set()
subprocess.run(
    ['gsutil', '-m', '-q', 'rsync', '-r',
     OUTPUT_DIR + '/', GCS_CHECKPOINT_PREFIX + '/' + RUN_ID + '/'],
    capture_output=True, timeout=180
)
print(f'Final checkpoint synced → {GCS_CHECKPOINT_PREFIX}/{RUN_ID}/')
train_wall_end = time.time()
TRAINING_ELAPSED_SEC = train_wall_end - train_wall_start

print()
print('=' * 70)
print(f'Exit code        : {proc.returncode}')
print(f'Total wall time  : {TRAINING_ELAPSED_SEC/60:.1f} min  ({TRAINING_ELAPSED_SEC:.0f}s)')
if epoch_times:
    print(f'Epochs completed : {len(epoch_times)}')
    print(f'Avg epoch time   : {sum(epoch_times)/len(epoch_times):.1f}s')
    print(f'Fastest epoch    : {min(epoch_times):.1f}s')
    print(f'Slowest epoch    : {max(epoch_times):.1f}s')
print('=' * 70)

if proc.returncode != 0:
    print('TRAINING FAILED — last lines:')
    for l in suppressed_lines[-60:]:
        print(l)
    raise RuntimeError(f'Training exited with code {proc.returncode}')

## Phase 2: Training Cost Report

In [ ]:
# ── 6. Cost report ─────────────────────────────────────────────────────────
import subprocess, time

# GPU utilisation snapshot
gpu_util = {}
try:
    r = subprocess.run(
        ['nvidia-smi',
         '--query-gpu=utilization.gpu,utilization.memory,memory.used,memory.total,power.draw',
         '--format=csv,noheader'],
        capture_output=True, text=True, timeout=5
    )
    if r.returncode == 0:
        parts = [p.strip() for p in r.stdout.strip().split(',')]
        gpu_util = {
            'gpu_utilisation':    parts[0] if len(parts) > 0 else '?',
            'mem_utilisation':    parts[1] if len(parts) > 1 else '?',
            'mem_used':           parts[2] if len(parts) > 2 else '?',
            'mem_total':          parts[3] if len(parts) > 3 else '?',
            'power_draw_w':       parts[4] if len(parts) > 4 else '?',
        }
except Exception:
    gpu_util = {}

elapsed_hr   = TRAINING_ELAPSED_SEC / 3600
compute_units = elapsed_hr * CU_RATE
total_session_elapsed = time.time() - TRAINING_START_WALL

cost_report = {
    'gpu_name':              gpu_info.get('name', 'unknown'),
    'gpu_tier':              gpu_tier,
    'cu_rate_per_hr':        CU_RATE,
    'training_elapsed_sec':  round(TRAINING_ELAPSED_SEC, 1),
    'training_elapsed_min':  round(TRAINING_ELAPSED_SEC / 60, 2),
    'training_elapsed_hr':   round(elapsed_hr, 4),
    'estimated_compute_units': round(compute_units, 3),
    'epochs_completed':      len(epoch_times),
    'epoch_times_sec':       [round(t, 1) for t in epoch_times],
    'avg_epoch_sec':         round(sum(epoch_times)/len(epoch_times), 1) if epoch_times else None,
    'gpu_snapshot':          gpu_util,
}

import json, os
cost_path = os.path.join(OUTPUT_DIR, f'training_cost_{RUN_ID}.json')
json.dump(cost_report, open(cost_path, 'w'), indent=2)

print('=' * 70)
print('TRAINING COST REPORT')
print('=' * 70)
print(f'  GPU                    : {gpu_info.get("name", "unknown")}  ({gpu_tier})')
print(f'  Training wall time     : {TRAINING_ELAPSED_SEC/60:.1f} min')
print(f'  Colab compute units    : {compute_units:.3f}  CU  ({CU_RATE} CU/hr × {elapsed_hr:.4f} hr)')
print(f'  Epochs completed       : {len(epoch_times)}')
if epoch_times:
    print(f'  Avg epoch time         : {sum(epoch_times)/len(epoch_times):.1f}s')
    print(f'  Per-epoch times (s)    : {[round(t,1) for t in epoch_times]}')
print()
if gpu_util:
    print('  Post-training GPU snapshot:')
    for k, v in gpu_util.items():
        print(f'    {k:25s}: {v}')
print(f'\n  Cost report saved → {cost_path}')
print('=' * 70)

## Phase 3: Artifacts & Metadata

In [ ]:
# ── 7. Artifacts + metadata ─────────────────────────────────────────────────
import os, json

artifacts = sorted(os.listdir(OUTPUT_DIR))

print('=' * 70)
print('ARTIFACTS')
print('=' * 70)
for f in artifacts:
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    size_str = f'{size/(1024*1024):.2f} MB' if size > 1024*1024 else f'{size/1024:.1f} KB'
    print(f'  {f:60s}  {size_str:>10}')

meta_files = [f for f in artifacts if f.startswith('training_metadata')]
if meta_files:
    meta = json.load(open(os.path.join(OUTPUT_DIR, meta_files[0])))
    print()
    print('TRAINING METADATA')
    print('-' * 70)
    core_keys = [
        'model_name', 'training_regime', 'objective_set',
        'epochs_completed', 'epochs_requested',
        'train_examples', 'val_examples',
        'hidden_dim', 'depth', 'dropout',
        'sequence_vocab_size', 'sequence_hidden_dim', 'max_seq_len',
        'value_weight', 'sequence_weight',
    ]
    for k in core_keys:
        if k in meta:
            print(f'  {k:35s}: {meta[k]}')
    print()
    print('  History encoder fields:')
    history_keys = [
        'use_history', 'history_turns', 'history_events_per_turn',
        'history_embed_dim', 'history_lstm_dim',
    ]
    for k in history_keys:
        print(f'    {k:30s}: {meta.get(k, "(missing)")}')

In [ ]:
# ── 8. Rebuild model from metadata + load weights ──────────────────────────
import os, sys, json
sys.path.insert(0, '/content/Pokemon-Showdown-Agents-Go-Brrrr')
import tensorflow as tf
from core.EntityModelV1 import build_entity_action_models
from core.StateVectorization import turn_outcome_dim

artifacts = sorted(os.listdir(OUTPUT_DIR))
meta_files           = [f for f in artifacts if f.startswith('training_metadata')]
training_model_files = [f for f in artifacts if f.startswith('training_model')]
policy_files         = [f for f in artifacts if f.endswith('.keras') and 'training_model' not in f and not f.startswith('policy_value')]
seq_vocab_files      = [f for f in artifacts if 'sequence_vocab' in f]
policy_vocab_files   = [f for f in artifacts if 'policy_vocab' in f and f.endswith('.json')]

if not meta_files:
    raise RuntimeError('No training_metadata file found.')

meta         = json.load(open(os.path.join(OUTPUT_DIR, meta_files[0])))
predict_seq  = 'sequence_auxiliary' in meta.get('objective_set', [])
predict_val  = 'value' in meta.get('objective_set', [])
predict_hist = 'history_encoder' in meta.get('objective_set', [])
predict_trans= 'transition' in meta.get('objective_set', [])

seq_vocab = None
if seq_vocab_files:
    seq_vocab = json.load(open(os.path.join(OUTPUT_DIR, seq_vocab_files[0])))

training_model, policy_model, _, history_attention_model = build_entity_action_models(
    vocab_sizes              = meta['entity_token_vocab_sizes'],
    num_policy_classes       = meta['num_action_classes'],
    hidden_dim               = meta['hidden_dim'],
    depth                    = meta['depth'],
    dropout                  = meta['dropout'],
    learning_rate            = meta['learning_rate'],
    token_embed_dim          = meta.get('token_embed_dim', 24),
    transition_dim           = turn_outcome_dim() if predict_trans else None,
    action_context_vocab_size= meta.get('num_action_context_classes') or None,
    action_embed_dim         = meta.get('action_embed_dim', 16),
    transition_weight        = meta.get('transition_weight', 0.25),
    predict_value            = predict_val,
    value_hidden_dim         = meta.get('value_hidden_dim') or None,
    value_weight             = meta.get('value_weight', 0.25),
    predict_sequence         = predict_seq,
    sequence_vocab_size      = meta.get('sequence_vocab_size') or None,
    sequence_hidden_dim      = meta.get('sequence_hidden_dim', 128),
    sequence_weight          = meta.get('sequence_weight', 0.1),
    max_seq_len              = meta.get('max_seq_len', 32),
    use_history              = predict_hist,
    history_vocab_size       = len(seq_vocab) if (predict_hist and seq_vocab) else None,
    history_embed_dim        = meta.get('history_embed_dim', 32),
    history_lstm_dim         = meta.get('history_lstm_dim', 64),
    history_turns            = meta.get('history_turns', 8),
    history_events_per_turn  = meta.get('history_events_per_turn', 24),
)

use_training_model = bool(training_model_files)
weights_file = training_model_files[0] if use_training_model else policy_files[0]
weights_path = os.path.join(OUTPUT_DIR, weights_file)
load_target  = training_model if use_training_model else policy_model
load_target.load_weights(weights_path)

print('Model loaded successfully.')
print('Training model outputs :', list(training_model.output_names) if use_training_model else ['policy'])
print('Attention extractor     :', 'available' if history_attention_model is not None else 'not available')
load_target.summary()

## Phase 4: Attention Weight Analysis (Semi-Observed Execution)

In [ ]:
# ── 9. Load validation examples for attention analysis ─────────────────────
import json, glob, os, sys, random
import numpy as np
sys.path.insert(0, '/content/Pokemon-Showdown-Agents-Go-Brrrr')

from core.BattleStateTracker import BattleStateTracker
from core.EntityTensorization import (
    build_entity_training_bundle, vectorize_entity_multitask_dataset
)

# Reload battle data (sample for analysis)
ANALYSIS_BATTLES = 100
json_files = sorted(glob.glob(os.path.join(data_path, '*.json')))
random.seed(42)
sample_files = random.sample(json_files, min(ANALYSIS_BATTLES, len(json_files)))

tracker = BattleStateTracker(form_change_species={'Palafin'}, history_turns=HISTORY_TURNS)
examples = []
for fpath in sample_files:
    try:
        battle = json.load(open(fpath))
        examples.extend(tracker.iter_turn_examples(battle, player='p1', include_switches=True))
    except Exception:
        pass

print(f'Loaded {len(examples):,} examples from {len(sample_files)} battles')
hist_counts = [len(ex.get('past_turn_events', [])) for ex in examples]
print(f'History buffer fill: avg={sum(hist_counts)/max(1,len(hist_counts)):.2f}  '
      f'max={max(hist_counts) if hist_counts else 0}')
print(f'Examples with full history (K={HISTORY_TURNS}): '
      f'{sum(1 for c in hist_counts if c >= HISTORY_TURNS)} / {len(examples)}')

In [ ]:
# ── 10. Vectorize + extract attention weights ──────────────────────────────
import numpy as np
from core.EntityTensorization import vectorize_entity_multitask_dataset, build_entity_training_bundle

# Reload vocabs from saved artifacts
policy_vocab_path = os.path.join(OUTPUT_DIR, [f for f in artifacts if 'policy_vocab' in f and f.endswith('.json')][0])
token_vocab_path  = os.path.join(OUTPUT_DIR, [f for f in artifacts if 'entity_token_vocab' in f][0])
seq_vocab_path    = os.path.join(OUTPUT_DIR, seq_vocab_files[0]) if seq_vocab_files else None

policy_vocab = json.load(open(policy_vocab_path))
token_vocabs = json.load(open(token_vocab_path))
seq_vocab    = json.load(open(seq_vocab_path)) if seq_vocab_path and os.path.exists(seq_vocab_path) else {}

print('Vectorizing analysis examples...')
X_analysis, _ = vectorize_entity_multitask_dataset(
    examples[:500],
    policy_vocab         = policy_vocab,
    token_vocabs         = token_vocabs,
    action_context_vocab = None,
    include_switches     = True,
    include_history      = predict_hist,
    sequence_vocab       = seq_vocab if seq_vocab else None,
    history_turns        = HISTORY_TURNS,
    history_events_per_turn = HISTORY_EVENTS_PER_TURN,
)

n    = len(examples[:500])
X_np = {k: np.array(v) for k, v in X_analysis.items()}
if predict_seq or predict_hist:
    X_np['my_action']  = np.zeros(n, dtype=np.int32)
    X_np['opp_action'] = np.zeros(n, dtype=np.int32)

# Run attention extractor — separate model, no action inputs needed
attn_weights_raw = None
if predict_hist and history_attention_model is not None:
    X_attn = {k: v for k, v in X_np.items() if k not in {'my_action', 'opp_action'}}
    print(f'Running attention extractor on {n} examples...')
    attn_weights_raw = np.array(history_attention_model(X_attn, training=False))
    print(f'Attention weights shape: {attn_weights_raw.shape}  (expected [{n}, K={HISTORY_TURNS}])')
else:
    print('No attention extractor available (use_history=False or model not rebuilt).')

In [ ]:
# ── 11. Attention weight distribution ─────────────────────────────────────
import numpy as np
from collections import Counter

if attn_weights_raw is not None:
    attn_weights = attn_weights_raw  # [N, K]
    K = attn_weights.shape[1]

    hist_mask_arr = X_np.get('event_history_mask')  # [N, K]
    has_history   = np.any(hist_mask_arr > 0, axis=1) if hist_mask_arr is not None else np.ones(len(attn_weights), dtype=bool)
    attn_filtered = attn_weights[has_history]

    mean_attn_by_turn = attn_filtered.mean(axis=0)
    std_attn_by_turn  = attn_filtered.std(axis=0)
    eps     = 1e-9
    entropy = -(attn_filtered * np.log(attn_filtered + eps)).sum(axis=1)

    print('=' * 70)
    print('ATTENTION WEIGHT ANALYSIS')
    print('=' * 70)
    print(f'  Examples with history       : {has_history.sum()} / {len(attn_weights)}')
    print(f'  Turn index 0 = oldest, K-1 = most recent (K={K})')
    print()
    print('  Mean attention weight per turn position:')
    for i in range(K):
        bar_len  = int(mean_attn_by_turn[i] * 40)
        bar      = '█' * bar_len + '░' * (40 - bar_len)
        turn_rel = i - K
        print(f'    turn[{i}] ({turn_rel:+d})  {mean_attn_by_turn[i]:.4f} ± {std_attn_by_turn[i]:.4f}  |{bar}|')
    print()
    print(f'  Attention entropy: mean={entropy.mean():.3f}  std={entropy.std():.3f}')
    print(f'  (max possible entropy for K={K}: {-np.log(1/K):.3f} nats)')

    peak_turns = np.argmax(attn_filtered, axis=1)
    peak_dist  = Counter(peak_turns.tolist())
    print()
    print('  Peak attention turn distribution:')
    for i in range(K):
        count = peak_dist.get(i, 0)
        pct   = 100 * count / max(1, len(attn_filtered))
        print(f'    turn[{i}]: {count:5d}  ({pct:.1f}%)')

    attn_report = {
        'run_id'           : RUN_ID,
        'n_examples'       : int(has_history.sum()),
        'history_turns'    : K,
        'mean_attn_by_turn': mean_attn_by_turn.tolist(),
        'std_attn_by_turn' : std_attn_by_turn.tolist(),
        'entropy_mean'     : float(entropy.mean()),
        'entropy_std'      : float(entropy.std()),
        'peak_turn_dist'   : {str(k): v for k, v in peak_dist.items()},
    }
    attn_path = os.path.join(OUTPUT_DIR, f'attention_analysis_{RUN_ID}.json')
    import json
    json.dump(attn_report, open(attn_path, 'w'), indent=2)
    print(f'\n  Saved → {attn_path}')
    print('=' * 70)
else:
    print('Attention weights not available — skipping analysis.')

## Phase 5: Training Diagnostics

In [ ]:
# ── 12. Loss curves from epoch metrics CSV ─────────────────────────────────
import csv, os

metrics_files = [f for f in artifacts if f.startswith('training_metrics') and f.endswith('.csv')]
if metrics_files:
    metrics_path = os.path.join(OUTPUT_DIR, metrics_files[0])
    rows = list(csv.DictReader(open(metrics_path)))
    if rows:
        headers = list(rows[0].keys())
        print('=' * 70)
        print('EPOCH METRICS SUMMARY')
        print('=' * 70)
        # Print header + all rows
        loss_cols  = [h for h in headers if 'loss' in h.lower()]
        acc_cols   = [h for h in headers if 'accuracy' in h.lower() or 'acc' in h.lower()]
        print(f'  Epoch  |  ' + '  |  '.join(loss_cols[:5]))
        print('  ' + '-' * 65)
        for row in rows:
            epoch = row.get('epoch', '?')
            vals  = '  |  '.join(f'{float(row.get(c, 0)):.4f}' for c in loss_cols[:5])
            print(f'  {epoch:>5}  |  {vals}')

        # Best validation loss epoch
        val_loss_col = next((c for c in loss_cols if 'val' in c and 'total' not in c.lower()), None)
        if not val_loss_col:
            val_loss_col = next((c for c in loss_cols if 'val' in c), None)
        if val_loss_col:
            best_row  = min(rows, key=lambda r: float(r.get(val_loss_col, 9999)))
            print()
            print(f'  Best epoch ({val_loss_col}): epoch {best_row.get("epoch")} = {float(best_row.get(val_loss_col, 0)):.4f}')
        print('=' * 70)
else:
    print('No epoch metrics CSV found.')

In [ ]:
# ── 13. History buffer utilisation ─────────────────────────────────────────
import numpy as np

hist_counts = [len(ex.get('past_turn_events', [])) for ex in examples]
total       = len(hist_counts)
buckets     = {}
for i in range(HISTORY_TURNS + 1):
    buckets[i] = sum(1 for c in hist_counts if c == i)
full_hist   = buckets.get(HISTORY_TURNS, 0)
no_hist     = buckets.get(0, 0)

print('=' * 70)
print(f'HISTORY BUFFER UTILISATION  (K={HISTORY_TURNS})')
print('=' * 70)
print(f'  Total examples        : {total:,}')
print(f'  No history (turn 1)   : {no_hist:,}  ({100*no_hist/max(1,total):.1f}%)')
print(f'  Full history (K={HISTORY_TURNS})  : {full_hist:,}  ({100*full_hist/max(1,total):.1f}%)')
print(f'  Mean history depth    : {np.mean(hist_counts):.2f}  turns')
print()
print('  Distribution:')
for depth, count in sorted(buckets.items()):
    pct = 100 * count / max(1, total)
    bar = '█' * int(pct / 2)
    print(f'    depth={depth}  {count:6,}  ({pct:5.1f}%)  {bar}')
print('=' * 70)

## Phase 6: Push Artifacts to GCS

### GCS Auth Setup (one-time)

To avoid the `authenticate_user()` browser prompt every session, store the service account key as a **Colab secret** once:

1. In Colab, click the **🔑 Secrets** icon in the left sidebar
2. Add a secret named **`GCS_SA_KEY_JSON`**
3. Paste the full contents of the service account JSON key file (ask the project owner for `pokemon-battle-agent-service-account.json`)
4. Enable "Notebook access"

After that, the push cell below reads the key automatically — no browser prompt required.  
If the secret is absent it falls back to `authenticate_user()` so the notebook still works without setup.

In [ ]:
# ── 14. Push to GCS ────────────────────────────────────────────────────────
# Credentials: loaded from Colab secret GCS_SA_KEY_JSON (see setup cell above).
# Falls back to authenticate_user() if the secret is absent.
import json, os, glob
from google.cloud import storage

BUCKET     = 'artifacts-model-serving'
GCS_PREFIX = f'artifacts/entity_history_v1/{RUN_ID}'

credentials = None
try:
    from google.colab import userdata
    sa_json = userdata.get('GCS_SA_KEY_JSON')
    from google.oauth2 import service_account
    credentials = service_account.Credentials.from_service_account_info(
        json.loads(sa_json),
        scopes=['https://www.googleapis.com/auth/cloud-platform'],
    )
    print('Using service account credentials from Colab secret GCS_SA_KEY_JSON.')
except Exception as _sa_err:
    print(f'Secret not available ({type(_sa_err).__name__}) — falling back to authenticate_user().')
    from google.colab import auth
    auth.authenticate_user()

try:
    client     = storage.Client(project='pokemon-battle-agent-494015', credentials=credentials)
    bucket_obj = client.bucket(BUCKET)

    pushed = []
    for local_path in sorted(glob.glob(os.path.join(OUTPUT_DIR, '*'))):
        if not os.path.isfile(local_path):
            continue
        fname     = os.path.basename(local_path)
        blob_name = f'{GCS_PREFIX}/{fname}'
        bucket_obj.blob(blob_name).upload_from_filename(local_path)
        size_kb = os.path.getsize(local_path) / 1024
        pushed.append(blob_name)
        print(f'  ✓ {fname:60s}  ({size_kb:.0f} KB)')

    print(f'\nPushed {len(pushed)} files → gs://{BUCKET}/{GCS_PREFIX}/')
    print('\nNext steps:')
    print('  1. Run /gcs-push locally to sync model_registry.json')
    print('  2. Run /ps-reload entity_history_v1 to start the server')
    print('  3. Run /ps-benchmark entity_history_v1 1000 --bg for Cell B/C ablation')

except Exception as e:
    print(f'GCS push failed: {e}')
    print('Artifacts are at:', OUTPUT_DIR)

In [ ]:
# ── 15. Final summary ──────────────────────────────────────────────────────
import time

session_elapsed = time.time() - TRAINING_START_WALL

print('=' * 70)
print('SESSION SUMMARY')
print('=' * 70)
print(f'  Run ID                 : {RUN_ID}')
print(f'  GPU                    : {gpu_info.get("name", "unknown")}')
print(f'  Total session time     : {session_elapsed/60:.1f} min')
print(f'  Training time          : {TRAINING_ELAPSED_SEC/60:.1f} min')
print(f'  Compute units used     : {(session_elapsed/3600)*CU_RATE:.3f} CU  (session-level estimate)')
print()
print('  Ablation next steps:')
print('    Cell A: ps-benchmark entity_action_v2_20260409_1811  (baseline, no history)')
print('    Cell B: ps-benchmark entity_history_v1 --no-rerank  (training pressure only)')
print('    Cell C: ps-benchmark entity_history_v1 --capture-aux-log /tmp/cell_c.jsonl')
print('    Then  : scripts/analyze_sequence_influence.py --log /tmp/cell_c.jsonl')
print('=' * 70)